# EE-3.3: Transformers with taps and phase shifters — and the Y-bus of a real network

## Objectives

At the end of this notebook you should be able to:

- model a transformer with an **off-nominal tap ratio** and derive its four Y-bus entries
- use a **tap changer** to control a voltage, and check your model against pandapower — including a silent configuration trap
- explain how a **phase-shifting transformer** steers active power between parallel paths, and estimate the effect with the DC approximation
- **build the Y-bus of a network from data tables** (lines with capacitance, tapped transformers, shunts) and verify it entry by entry against pandapower's internal matrix

This is the last step of the foundations track before Module 04. EE-2.1 introduced the Y-bus for simple branches, EE-3.1 added per-unit and transformers at nominal ratio, EE-3.2 added line capacitance. Here every piece comes together: the Y-bus you build at the end is exactly the matrix a real power flow tool solves.

In [1]:
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore", category=FutureWarning)
S_BASE = 100.0                       # MVA
OMEGA = 2 * np.pi * 50.0


def numerical_jacobian(F, x, h=1e-7):
    """Central differences (M1.3)."""
    return np.column_stack([(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h])


def power_flow_pu(Y, S_inj, tol=1e-12):
    """Newton power flow in per-unit (EE-3.1): bus 0 is the slack at 1.0 pu, all others PQ."""
    n = len(Y)
    def F(x):
        V = np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        d = V * np.conj(Y @ V) - S_inj
        return np.r_[d.real[1:], d.imag[1:]]
    x = np.r_[np.zeros(n - 1), np.ones(n - 1)]
    for _ in range(40):
        if np.abs(F(x)).max() < tol:
            return np.r_[1.0, x[n - 1:]] * np.exp(1j * np.r_[0.0, x[:n - 1]])
        x = x + np.linalg.solve(numerical_jacobian(F, x), -F(x))
    raise RuntimeError("no convergence")

### A transformer with an off-nominal ratio

In EE-3.1 the transformer had its nominal ratio (400/110 kV), so in per-unit it became a plain series impedance. Real transformers have a **tap changer**: extra winding turns that can be switched in or out, typically ±10 % in steps of about 1.25 %. Tap position n gives the per-unit ratio t = 1 + n · step on the high-voltage side.

Model it as an ideal transformer t : 1 followed by the series admittance y. Working out the currents at both ends gives four Y-bus entries (f = high-voltage side, k = low-voltage side):

| Entry | Value |
|---|---|
| Y_ff | y / \|t\|² |
| Y_fk | −y / t\* |
| Y_kf | −y / t |
| Y_kk | y |

For a real t the matrix stays symmetric. With t = 1 it reduces to EE-2.1's line stamp. Line charging (EE-3.2) adds half the shunt susceptance at each end.

In [2]:
def stamp_branch(Y, f, k, y, t=1.0 + 0j, b_shunt=0.0):
    """Add a branch (series admittance y, tap t on side f, total line charging b_shunt) to the Y-bus."""
    Y[f, f] += (y + 0.5j * b_shunt) / abs(t) ** 2
    Y[f, k] += -y / np.conj(t)
    Y[k, f] += -y / t
    Y[k, k] += y + 0.5j * b_shunt

### A tap changer controls the voltage

Take EE-3.1's 400/110 kV transformer (250 MVA, v_k = 12 %, v_kr = 0.3 %) feeding a 110 kV line (1 + j4 Ω) and a load of 120 MW + 40 Mvar. Solve it with three tap positions in your own model and in pandapower — and compare pandapower's internal Y-bus with yours:

In [3]:
vk, vkr, sn = 0.12, 0.003, 250.0
z_trafo = (vkr + 1j * np.sqrt(vk**2 - vkr**2)) * S_BASE / sn
z_line_110 = (1 + 4j) / (110**2 / S_BASE)
STEP = 1.25                                   # percent per tap position


def tap_network_pandapower(tap_pos, tap_changer_type="Ratio"):
    net = pp.create_empty_network(sn_mva=S_BASE)          # pandapower's default base is 1 MVA
    b = [pp.create_bus(net, vn_kv=v) for v in [400, 110, 110]]
    pp.create_ext_grid(net, b[0], vm_pu=1.0)
    pp.create_transformer_from_parameters(net, b[0], b[1], sn_mva=sn, vn_hv_kv=400, vn_lv_kv=110, vk_percent=12,
                                          vkr_percent=0.3, pfe_kw=0, i0_percent=0, tap_side="hv", tap_neutral=0,
                                          tap_min=-10, tap_max=10, tap_step_percent=STEP, tap_pos=tap_pos,
                                          tap_changer_type=tap_changer_type)
    pp.create_line_from_parameters(net, b[1], b[2], length_km=1, r_ohm_per_km=1, x_ohm_per_km=4, c_nf_per_km=0, max_i_ka=3)
    pp.create_load(net, b[2], p_mw=120, q_mvar=40)
    pp.runpp(net, numba=False)
    return net


rows = []
for tap_pos in [-4, 0, 3]:
    Y = np.zeros((3, 3), dtype=complex)
    stamp_branch(Y, 0, 1, 1 / z_trafo, t=1 + tap_pos * STEP / 100)
    stamp_branch(Y, 1, 2, 1 / z_line_110)
    V = power_flow_pu(Y, np.array([0, 0, -(120 + 40j) / S_BASE]))
    net = tap_network_pandapower(tap_pos)
    Y_pp = net._ppc["internal"]["Ybus"].toarray()
    rows.append({"tap_pos": tap_pos, "ratio_pu": 1 + tap_pos * STEP / 100, "own_V_load_pu": abs(V[2]),
                 "pandapower_V_load_pu": net.res_bus.vm_pu[2], "max_Ybus_difference": np.abs(Y - Y_pp).max()})
taps = pd.DataFrame(rows)
assert np.allclose(taps.own_V_load_pu, taps.pandapower_V_load_pu, atol=1e-9) and (taps.max_Ybus_difference < 1e-12).all()
taps

   tap_pos  ratio_pu  own_V_load_pu  pandapower_V_load_pu  max_Ybus_difference
0       -4    0.9500       1.004956              1.004956         3.554448e-15
1        0    1.0000       0.949012              0.949012         0.000000e+00
2        3    1.0375       0.910282              0.910282         3.552714e-15

Your Y-bus equals pandapower's to rounding error, and so do the voltages. Four taps down (ratio 0.95) lift the load bus from 0.949 to 1.005 pu: a lower ratio on the high-voltage side means more volts per turn on the low-voltage side. Tap changers that can move under load (OLTCs) keep distribution voltages in their band all day, typically stepping every few minutes as demand changes.

**A trap worth seeing once.** pandapower 3 needs the tap changer's type (`tap_changer_type="Ratio"`). Leave it out and the tap position is silently ignored — no error, no warning:

In [4]:
ignored = [tap_network_pandapower(p, tap_changer_type=None).res_bus.vm_pu[2] for p in [-4, 0, 3]]
print(f"without tap_changer_type: load voltage at taps -4, 0, +3 = {np.round(ignored, 6)}")
assert np.ptp(ignored) < 1e-12, 'all three are identical: the tap had no effect'

without tap_changer_type: load voltage at taps -4, 0, +3 = [0.949012 0.949012 0.949012]


Three identical results. Nothing in pandapower's output reveals the problem; only an independent model — or a sanity check that "changing the tap should change the voltage" — catches it. This is why every lesson in this course validates against something else: tools are only as right as their configuration.

### Phase-shifting transformers steer power

A **phase-shifting transformer** (PST) has a complex ratio t = e^{jφ}: it shifts the voltage angle by φ without changing its magnitude. In a radial network that only rotates all downstream angles. In a **meshed** network it changes how power splits between parallel paths — because active power follows angle differences (EE-2.2).

Two 150 km, 400 kV lines connect a slack to a 1,000 MW load; one of them runs through a 1,000 MVA PST (v_k = 10 %):

In [5]:
Z_BASE_400 = 400**2 / S_BASE
z_A = z_B = (0.03 + 0.3j) * 150 / Z_BASE_400
z_pst = 0.10j * S_BASE / 1000


def pst_network(shift_deg):
    """Own model and pandapower for bus 0 (slack), bus 1 (load), bus 2 (between PST and line B)."""
    Y = np.zeros((3, 3), dtype=complex)
    stamp_branch(Y, 0, 1, 1 / z_A)
    stamp_branch(Y, 0, 2, 1 / z_pst, t=np.exp(1j * np.radians(shift_deg)))
    stamp_branch(Y, 2, 1, 1 / z_B)
    V = power_flow_pu(Y, np.array([0, -10.0, 0]))
    flow_A = (V[0] * np.conj((V[0] - V[1]) / z_A)).real * S_BASE
    net = pp.create_empty_network(sn_mva=S_BASE)
    b = [pp.create_bus(net, vn_kv=400) for _ in range(3)]
    pp.create_ext_grid(net, b[0])
    pp.create_line_from_parameters(net, b[0], b[1], 150, 0.03, 0.3, 0, 3)
    pp.create_transformer_from_parameters(net, b[0], b[2], sn_mva=1000, vn_hv_kv=400, vn_lv_kv=400, vk_percent=10,
                                          vkr_percent=0, pfe_kw=0, i0_percent=0, shift_degree=shift_deg)
    pp.create_line_from_parameters(net, b[2], b[1], 150, 0.03, 0.3, 0, 3)
    pp.create_load(net, b[1], p_mw=1000, q_mvar=0)
    pp.runpp(net, numba=False)
    return flow_A, net.res_line.p_from_mw[0]


rows = [{"shift_deg": s, **dict(zip(["own_flow_line_A_MW", "pandapower_flow_line_A_MW"], pst_network(s)))}
        for s in [5, 0, -5, -10]]
pst = pd.DataFrame(rows)
assert np.allclose(pst.own_flow_line_A_MW, pst.pandapower_flow_line_A_MW, rtol=1e-8)
x_loop = (z_A + z_B + z_pst).imag
print(f"DC estimate: {np.radians(5) / x_loop * S_BASE:.0f} MW shifted per 5 degrees")
pst.round(1)

DC estimate: 132 MW shifted per 5 degrees


   shift_deg  own_flow_line_A_MW  pandapower_flow_line_A_MW
0          5               713.0                      713.0
1          0               584.8                      584.8
2         -5               457.9                      457.9
3        -10               333.4                      333.4

With no shift, line A carries 585 MW — more than half, because the other path also has the PST's impedance. A −5° shift moves 127 MW from line A to the PST path; −10° moves 251 MW. The DC approximation (EE-2.2) predicts the change from the loop reactance alone: Δ P ≈ φ / (x_A + x_B + x_PST) = 132 MW per 5°, within 4 % of the full AC result.

Grid operators install PSTs on interconnectors to control **loop flows** — power that takes an unscheduled route through a neighbour's grid because of how impedances split it — and to relieve congested lines without building new ones. In Module 04 you will see how flow-based market coupling and redispatch treat them.

### The Y-bus of a network from data tables

Real network data come as tables: buses, lines, transformers, shunts — exactly how pandapower stores them. Build a five-bus, two-voltage-level network from such tables, assemble its Y-bus with `stamp_branch`, and compare it entry by entry with the matrix pandapower solves:

- three 400 kV lines forming a ring (with capacitance, EE-3.2),
- a 400/110 kV transformer at tap position −4,
- a 40 km, 110 kV line to a load of 80 MW + 20 Mvar, another load of 300 MW + 80 Mvar on the 400 kV ring,
- a 50 Mvar shunt reactor at the transformer's 400 kV bus.

In [6]:
buses = pd.DataFrame({"vn_kv": [400, 400, 400, 110, 110]})
lines = pd.DataFrame({"from_bus": [0, 1, 0, 3], "to_bus": [1, 2, 2, 4], "length_km": [150, 120, 200, 40],
                      "r_ohm_per_km": [0.03, 0.03, 0.03, 0.12], "x_ohm_per_km": [0.3, 0.3, 0.3, 0.4],
                      "c_nf_per_km": [12, 12, 12, 9]})
trafos = pd.DataFrame({"hv_bus": [2], "lv_bus": [3], "sn_mva": [250], "vk_percent": [12], "vkr_percent": [0.3],
                       "tap_pos": [-4], "tap_step_percent": [STEP]})
shunts = pd.DataFrame({"bus": [2], "q_mvar": [50]})           # positive = reactor (absorbs Q at 1 pu)
loads = pd.DataFrame({"bus": [1, 4], "p_mw": [300, 80], "q_mvar": [80, 20]})


def build_ybus(buses, lines, trafos, shunts, s_base=S_BASE):
    """Per-unit Y-bus from pandapower-style tables."""
    Y = np.zeros((len(buses), len(buses)), dtype=complex)
    for l in lines.itertuples():
        z_base = buses.vn_kv[l.from_bus] ** 2 / s_base
        z = (l.r_ohm_per_km + 1j * l.x_ohm_per_km) * l.length_km / z_base
        b = OMEGA * l.c_nf_per_km * 1e-9 * l.length_km * z_base
        stamp_branch(Y, l.from_bus, l.to_bus, 1 / z, b_shunt=b)
    for tr in trafos.itertuples():
        vk_, vkr_ = tr.vk_percent / 100, tr.vkr_percent / 100
        z = (vkr_ + 1j * np.sqrt(vk_**2 - vkr_**2)) * s_base / tr.sn_mva
        stamp_branch(Y, tr.hv_bus, tr.lv_bus, 1 / z, t=1 + tr.tap_pos * tr.tap_step_percent / 100)
    for sh in shunts.itertuples():
        Y[sh.bus, sh.bus] += -1j * sh.q_mvar / s_base
    return Y


def pandapower_from_tables(buses, lines, trafos, shunts, loads):
    net = pp.create_empty_network(sn_mva=S_BASE)
    for v in buses.vn_kv:
        pp.create_bus(net, vn_kv=v)
    pp.create_ext_grid(net, 0, vm_pu=1.0)
    for l in lines.itertuples():
        pp.create_line_from_parameters(net, l.from_bus, l.to_bus, l.length_km, l.r_ohm_per_km, l.x_ohm_per_km,
                                       l.c_nf_per_km, max_i_ka=3)
    for tr in trafos.itertuples():
        pp.create_transformer_from_parameters(net, tr.hv_bus, tr.lv_bus, sn_mva=tr.sn_mva,
                                              vn_hv_kv=buses.vn_kv[tr.hv_bus], vn_lv_kv=buses.vn_kv[tr.lv_bus],
                                              vk_percent=tr.vk_percent, vkr_percent=tr.vkr_percent, pfe_kw=0,
                                              i0_percent=0, tap_side="hv", tap_neutral=0, tap_min=-10, tap_max=10,
                                              tap_step_percent=tr.tap_step_percent, tap_pos=tr.tap_pos,
                                              tap_changer_type="Ratio")
    for sh in shunts.itertuples():
        pp.create_shunt(net, sh.bus, q_mvar=sh.q_mvar)
    for ld in loads.itertuples():
        pp.create_load(net, ld.bus, p_mw=ld.p_mw, q_mvar=ld.q_mvar)
    pp.runpp(net, numba=False)
    return net


Y5 = build_ybus(buses, lines, trafos, shunts)
net5 = pandapower_from_tables(buses, lines, trafos, shunts, loads)
order = net5._pd2ppc_lookups["bus"]                           # pandapower's internal bus order
Y5_pp = net5._ppc["internal"]["Ybus"].toarray()[np.ix_(order, order)]
print(f"largest difference between the two Y-bus matrices: {np.abs(Y5 - Y5_pp).max():.1e} pu")
print(f"non-zero entries: {np.count_nonzero(Y5)} of {Y5.size}")

largest difference between the two Y-bus matrices: 8.9e-16 pu
non-zero entries: 15 of 25


The two 5 × 5 complex matrices agree to rounding error — line charging, tap ratio, shunt reactor and per-unit conversion across two voltage levels included. Only 15 of 25 entries are non-zero: each bus connects to a few neighbours. In a real transmission grid with a thousand buses or more, over 99 % of the entries are zero, which is why tools store the Y-bus as a sparse matrix (M1.3).

Finally, solve the network with your own power flow:

In [7]:
S_inj = np.zeros(5, dtype=complex)
for ld in loads.itertuples():
    S_inj[ld.bus] -= (ld.p_mw + 1j * ld.q_mvar) / S_BASE
V5 = power_flow_pu(Y5, S_inj)
result = pd.DataFrame({"vn_kv": buses.vn_kv, "own_vm_pu": np.abs(V5), "pandapower_vm_pu": net5.res_bus.vm_pu,
                       "own_va_deg": np.angle(V5, deg=True), "pandapower_va_deg": net5.res_bus.va_degree})
assert np.allclose(result.own_vm_pu, result.pandapower_vm_pu, atol=1e-9)
assert np.allclose(result.own_va_deg, result.pandapower_va_deg, atol=1e-7)
assert result.own_vm_pu.between(0.95, 1.05).all(), "all voltages inside the usual operating band"
result.round(5)

   vn_kv  own_vm_pu  pandapower_vm_pu  own_va_deg  pandapower_va_deg
0    400    1.00000           1.00000     0.00000            0.00000
1    400    0.99284           0.99284    -3.89905           -3.89905
2    400    0.99579           0.99579    -3.11686           -3.11686
3    110    1.03347           1.03347    -5.20241           -5.20241
4    110    0.96936           0.96936   -10.82240          -10.82240

Identical to pandapower at every bus, and every voltage is inside the 0.95–1.05 pu band thanks to the tap at −4. You have now built — from first principles — every element of an AC power flow: branch models, transformers, per-unit, the Y-bus and the Newton solver. Module 04 scales it up to real grids, adds the full analytical Jacobian, and compares it with the DC power flow you met in EE-2.2.

## Check your understanding

**Taps and phase shifters**

1. Why does a *lower* ratio on the high-voltage side *raise* the low-voltage side's voltage? What happens to the reactive power the transformer draws from the 400 kV side?
2. Why is the Y-bus no longer symmetric when the transformer has a phase shift — and what does that mean for the "reciprocity" of the network?

**Data and validation**

1. In the five-bus network, which entries of the Y-bus change when the tap moves from −4 to −3? Which stay the same?
2. pandapower silently ignored the tap without `tap_changer_type`. Name two other silent configuration errors that would make a power flow wrong but still converge, and a check that would catch each.